# 07 Prediction Testing on Unseen Images

## Objective
Test predictions on unseen test images:
- ✓ Load best model
- ✓ Make predictions on test set
- ✓ Display predictions with confidence scores
- ✓ Show misclassified examples
- ✓ Analyze prediction confidence distribution
- ✓ Generate prediction report

**Prerequisites:** All previous notebooks (01-06)

### Import Libraries

import sys
import os

sys.path.insert(0, '../utils')

from model_utils import load_model
from visualization_utils import plot_predictions_with_confidence

import numpy as np
import json
import matplotlib.pyplot as plt
import pandas as pd

print("✓ Libraries imported successfully!")

### Load Data and Best Model

print("Loading test data...\n")

# Load test data
X_test = np.load('../results/preprocessed_data/X_test.npy')
y_test = np.load('../results/preprocessed_data/y_test.npy')

# Load label encoding
with open('../results/preprocessed_data/label_encoding.json', 'r') as f:
    label_encoding = json.load(f)

num_classes = len(label_encoding)
reverse_encoding = {v: k for k, v in label_encoding.items()}
class_names = [reverse_encoding[i] for i in range(num_classes)]

print(f"✓ Test data loaded: {X_test.shape}")

# Load best model
print("\nLoading best model...\n")

with open('../results/reports/best_model.json', 'r') as f:
    best_model_info = json.load(f)

best_model_name = best_model_info['name']
print(f"Best model: {best_model_name}")
print(f"  F1-Score: {best_model_info['metrics']['f1_score']:.4f}")

if 'mobilenet' in best_model_name.lower():
    model_path = '../models/mobilenet/mobilenet_v2.h5'
else:
    model_path = '../models/cnn/cnn_baseline.h5'

model = load_model(model_path)
print(f"\n✓ Model loaded from: {model_path}")

### 1. Make Predictions

print("\nMaking predictions on test set...\n")

# Predict
predictions = model.predict(X_test, verbose=0)
pred_labels_idx = np.argmax(predictions, axis=1)
true_labels_idx = np.argmax(y_test, axis=1)

# Convert indices to class names
pred_labels = np.array([class_names[i] for i in pred_labels_idx])
true_labels = np.array([class_names[i] for i in true_labels_idx])

# Get confidence scores
confidence = np.max(predictions, axis=1)

# Calculate accuracy
accuracy = np.mean(pred_labels == true_labels)

print(f"✓ Predictions completed")
print(f"  Test Set Size: {len(X_test)}")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Mean Confidence: {confidence.mean():.4f}")
print(f"  Min Confidence: {confidence.min():.4f}")
print(f"  Max Confidence: {confidence.max():.4f}")

### 2. Analyze Prediction Confidence

print("\n" + "="*70)
print("CONFIDENCE ANALYSIS")
print("="*70 + "\n")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence distribution
axes[0].hist(confidence, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].axvline(confidence.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {confidence.mean():.3f}')
axes[0].set_xlabel('Confidence', fontweight='bold')
axes[0].set_ylabel('Frequency', fontweight='bold')
axes[0].set_title('Prediction Confidence Distribution', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Correct vs Incorrect confidence
correct_mask = (pred_labels == true_labels)
correct_confidence = confidence[correct_mask]
incorrect_confidence = confidence[~correct_mask]

axes[1].hist(correct_confidence, bins=20, alpha=0.7, label=f'Correct (n={len(correct_confidence)})', color='green')
axes[1].hist(incorrect_confidence, bins=20, alpha=0.7, label=f'Incorrect (n={len(incorrect_confidence)})', color='red')
axes[1].set_xlabel('Confidence', fontweight='bold')
axes[1].set_ylabel('Frequency', fontweight='bold')
axes[1].set_title('Confidence: Correct vs Incorrect Predictions', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/plots/prediction_confidence.png', dpi=300, bbox_inches='tight')
print("✓ Saved: prediction_confidence.png")

### 3. Display Correct Predictions

print("\nShowing correct predictions...\n")

# Get correct predictions
correct_indices = np.where(correct_mask)[0]
sample_correct_idx = np.random.choice(correct_indices, min(8, len(correct_indices)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
axes = axes.flatten()

for plot_idx, img_idx in enumerate(sample_correct_idx):
    ax = axes[plot_idx]
    img = X_test[img_idx]
    
    ax.imshow(img)
    ax.set_title(f"True: {true_labels[img_idx]}\nPred: {pred_labels[img_idx]}\nConf: {confidence[img_idx]:.3f}",
                 fontweight='bold', color='green', fontsize=9)
    ax.axis('off')

plt.suptitle('Correct Predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/correct_predictions.png', dpi=300, bbox_inches='tight')
print("✓ Saved: correct_predictions.png")

### 4. Display Incorrect Predictions

print("\nShowing incorrect predictions (misclassifications)...\n")

# Get incorrect predictions
incorrect_indices = np.where(~correct_mask)[0]

if len(incorrect_indices) > 0:
    sample_incorrect_idx = np.random.choice(incorrect_indices, min(8, len(incorrect_indices)), replace=False)
    
    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    axes = axes.flatten()
    
    for plot_idx, img_idx in enumerate(sample_incorrect_idx):
        ax = axes[plot_idx]
        img = X_test[img_idx]
        
        ax.imshow(img)
        ax.set_title(f"True: {true_labels[img_idx]}\nPred: {pred_labels[img_idx]}\nConf: {confidence[img_idx]:.3f}",
                     fontweight='bold', color='red', fontsize=9)
        ax.axis('off')
    
    plt.suptitle('Misclassified Predictions', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/plots/misclassified_predictions.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: misclassified_predictions.png")
    else:
    print("✓ No misclassifications found!")

### 5. Per-Class Prediction Statistics

print("\n" + "="*70)
print("PER-CLASS PREDICTION STATISTICS")
print("="*70 + "\n")

class_stats = []
for class_idx, class_name in enumerate(class_names):
    class_mask = (true_labels_idx == class_idx)
    
    if class_mask.sum() > 0:
        class_correct = (pred_labels_idx[class_mask] == class_idx).sum()
        class_accuracy = class_correct / class_mask.sum()
        class_avg_confidence = confidence[class_mask].mean()
        
        class_stats.append({
            'Class': class_name,
            'Samples': int(class_mask.sum()),
            'Correct': int(class_correct),
            'Accuracy': f"{class_accuracy:.4f}",
            'Avg_Confidence': f"{class_avg_confidence:.4f}"
        })

class_stats_df = pd.DataFrame(class_stats).sort_values('Accuracy', ascending=False)
print(class_stats_df.to_string(index=False))

# Save stats
class_stats_df.to_csv('../results/reports/per_class_prediction_stats.csv', index=False)
print("\n✓ Saved: per_class_prediction_stats.csv")

### 6. Low Confidence Predictions

print("\n" + "="*70)
print("LOW CONFIDENCE PREDICTIONS")
print("="*70 + "\n")

# Get predictions with confidence < 0.7
low_conf_threshold = 0.7
low_conf_mask = (confidence < low_conf_threshold)
low_conf_count = low_conf_mask.sum()
low_conf_percentage = low_conf_count / len(confidence) * 100

print(f"Predictions with confidence < {low_conf_threshold}: {low_conf_count} ({low_conf_percentage:.2f}%)")

if low_conf_count > 0:
    low_conf_indices = np.where(low_conf_mask)[0]
    low_conf_accuracy = (pred_labels[low_conf_indices] == true_labels[low_conf_indices]).sum() / low_conf_count
    print(f"Accuracy on low confidence predictions: {low_conf_accuracy:.4f}")
    
    # Sample 8 low confidence predictions
    sample_low_conf_idx = np.random.choice(low_conf_indices, min(8, len(low_conf_indices)), replace=False)
    
    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    axes = axes.flatten()
    
    for plot_idx, img_idx in enumerate(sample_low_conf_idx):
        ax = axes[plot_idx]
        img = X_test[img_idx]
        
        ax.imshow(img)
        is_correct = "✓" if pred_labels[img_idx] == true_labels[img_idx] else "✗"
        ax.set_title(f"{is_correct} True: {true_labels[img_idx]}\nPred: {pred_labels[img_idx]}\nConf: {confidence[img_idx]:.3f}",
                     fontweight='bold', color='orange', fontsize=9)
        ax.axis('off')
    
    plt.suptitle(f'Low Confidence Predictions (< {low_conf_threshold})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/plots/low_confidence_predictions.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: low_confidence_predictions.png")

### 7. Generate Prediction Report

print("\n" + "="*70)
print("GENERATING PREDICTION REPORT")
print("="*70 + "\n")

prediction_report = {
    'model': best_model_name,
    'test_set': {
        'total_samples': int(len(X_test)),
        'num_classes': int(num_classes)
    },
    'overall_performance': {
        'accuracy': float(accuracy),
        'mean_confidence': float(confidence.mean()),
        'min_confidence': float(confidence.min()),
        'max_confidence': float(confidence.max()),
        'std_confidence': float(confidence.std()),
        'correct_predictions': int(correct_mask.sum()),
        'incorrect_predictions': int((~correct_mask).sum())
    },
    'confidence_thresholds': {
        'high_confidence_0.9': int((confidence >= 0.9).sum()),
        'high_confidence_0.8': int((confidence >= 0.8).sum()),
        'medium_confidence_0.7': int((confidence >= 0.7).sum()),
        'low_confidence_below_0.7': int((confidence < 0.7).sum())
    }
}

with open('../results/reports/prediction_report.json', 'w') as f:
    json.dump(prediction_report, f, indent=4)

print("✓ Prediction report saved")
print(json.dumps(prediction_report, indent=2))

### 8. Summary

print("\n" + "="*70)
print("✓ PREDICTION TESTING COMPLETE")
print("="*70)

print(f"\n📊 Results Summary:")
print(f"  Model: {best_model_name}")
print(f"  Test Accuracy: {accuracy:.4f}")
print(f"  Correct Predictions: {correct_mask.sum()} / {len(X_test)}")
print(f"  Mean Confidence: {confidence.mean():.4f}")

print(f"\n💾 Files Saved:")
print(f"  - prediction_report.json")
print(f"  - per_class_prediction_stats.csv")
print(f"  - Visualization plots")

print(f"\n→ Next Step: 08_tflite_conversion.ipynb")